In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
#%% md
# Фильтрация и дедупликация новостей

#Сравниваются четыре гипотезы: точный дубль, дубль после нормализации, почти одинаковый текст (TF-IDF) и семантический дубль с **ModernBERT**. Кластеры формируются только из публикаций, вышедших не далее чем через 14 дней друг от друга.
#%%


from pathlib import Path
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.auto import tqdm

# ИСПРАВЛЕНО: раньше DATA_PATH указывал на директорию, а не на CSV-файл,
# из-за чего pd.read_csv падал с IsADirectoryError.
DATA_PATH = '/kaggle/input/datasets/artemwalker/telegram-news/telegram_news_raw.csv'
OUT_DIR = Path('deduplication_results')
OUT_DIR.mkdir(exist_ok=True)
MODEL_NAME = 'deepvk/RuModernBERT-small'
WINDOW_DAYS = 14
pd.set_option('display.max_colwidth', 180)

In [ ]:
import torch
# ИСПРАВЛЕНО: torch.cuda.get_device_name(0) падает, если CUDA недоступна.
if torch.cuda.is_available():
    print(True)
    print(torch.cuda.get_device_name(0))  # Покажет модель GPU
else:
    print(False)
    print('CUDA недоступна — вычисления пойдут на CPU')

In [ ]:
#%% md
## 1. Загрузка данных
#Читаем CSV и проверяем объём и временной диапазон данных.
#%%
df = pd.read_csv(DATA_PATH, parse_dates=['post_dttm']).reset_index(drop=True)
print(f'Новостей: {len(df):,}; период: {df.post_dttm.min():%d.%m.%Y}—{df.post_dttm.max():%d.%m.%Y}')
df[['id', 'source', 'post_dttm', 'data']].head(3)

In [ ]:
## 2. Нормализация текста
#Удаляем URL, упоминания каналов, различия регистра и лишнюю пунктуацию.
#%%
def normalize_text(text):
    """Удаляет технические различия, не удаляя факты, числа и тикеры."""
    text = str(text or '').lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'@[\w_]+', ' ', text)
    text = re.sub(r'[^\w\s%.-]', ' ', text, flags=re.UNICODE)
    return re.sub(r'\s+', ' ', text).strip()

df['text_normalized'] = df['data'].fillna('').map(normalize_text)
df['text_length'] = df['text_normalized'].str.len()
# Пустые тексты не считаем дублями друг друга.
df = df[df.text_length.gt(0)].copy().reset_index(drop=True)
len(df)

In [ ]:
## 3. Формирование кластеров
#Объединяем связанные пары похожих новостей в кластеры и считаем метрики экспериментов.
#%%
class UnionFind:
    def __init__(self, n): self.parent = list(range(n))
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b: self.parent[b] = a

def clusters_from_edges(n, edges):
    uf = UnionFind(n)
    for a, b in edges: uf.union(a, b)
    return pd.factorize([uf.find(i) for i in range(n)])[0]

def exact_clusters(series):
    return pd.factorize(series)[0]

def top_k_edges(vectors, dates, threshold, n_neighbors=6, batch_size=128):
    """бачевый top-K поиск без матрицы похожести N×N в памяти (19220x19220)много расчетов """
    ts = pd.to_datetime(dates, utc=True).astype('int64').to_numpy()
    max_delta = pd.Timedelta(days=WINDOW_DAYS).value
    edges = []
    for start in tqdm(range(0, len(ts), batch_size), desc=f'Поиск пар ≥ {threshold:.2f}', leave=False):
        stop = min(start + batch_size, len(ts))
        sim = vectors[start:stop] @ vectors.T
        if hasattr(sim, 'toarray'): sim = sim.toarray()
        idx = np.argpartition(sim, -n_neighbors, axis=1)[:, -n_neighbors:]
        for local_i, candidates in enumerate(idx):
            i = start + local_i
            for j in candidates:
                if i < j and sim[local_i, j] >= threshold and abs(ts[i] - ts[j]) <= max_delta:
                    edges.append((i, int(j)))
    return edges

def method_stats(name, clusters):
    sizes = pd.Series(clusters).value_counts()
    removed = int((sizes - 1).clip(lower=0).sum())
    return dict(method=name, duplicate_groups=int((sizes > 1).sum()),
                duplicate_posts=int(sizes[sizes > 1].sum()), posts_to_remove=removed,
                remaining_posts=len(df) - removed, largest_group=int(sizes.max()))

In [ ]:
#%% md
## 4. Базовые гипотезы дедупликации
#Сравниваем точные совпадения, нормализованные совпадения и почти идентичные тексты через TF-IDF.
#%%

experiments = {}
experiments['exact_raw'] = exact_clusters(df.data.fillna(''))
experiments['exact_normalized'] = exact_clusters(df.text_normalized)

# символьный TF-IDF устойчив к небольшим правкам текста. почитай за analyzer
tfidf = TfidfVectorizer(analyzer='word', ngram_range=(2, 5), min_df=4, max_features=500_000)
print('Строю TF-IDF-признаки…')
tfidf_vectors = tfidf.fit_transform(tqdm(df.text_normalized, desc='TF-IDF: подготовка текстов'))
for threshold in tqdm((0.90, 0.95), desc='TF-IDF: пороги'):
    name = f'tfidf_{threshold:.2f}'
    experiments[name] = clusters_from_edges(
        len(df), top_k_edges(tfidf_vectors, df.post_dttm, threshold)
    )

report = pd.DataFrame([method_stats(name, clusters) for name, clusters in experiments.items()])
report.sort_values('posts_to_remove', ascending=False)

In [ ]:
#%% md
## 5. Проверка найденных дублей
#Выбираем метод и просматриваем кандидатов на исключение до сохранения результата.
#%%
SELECTED_METHOD = 'tfidf_0.90'  # или exact_raw / exact_normalized / tfidf_0.95

# Добавляем номера кластеров к строкам датафрейма
df_view = df.assign(duplicate_cluster=experiments[SELECTED_METHOD])

# В каждом кластере сохраняем самый ранний пост,
# последующие — кандидаты на удаление
ordered = df_view.sort_values(['duplicate_cluster', 'post_dttm'])
posts_to_remove = ordered[
    ordered.duplicated('duplicate_cluster', keep='first')
].copy()

print(f'Метод: {SELECTED_METHOD}')
print(f'Постов к исключению: {len(posts_to_remove):,}')

display(
    posts_to_remove[
        ['duplicate_cluster', 'id', 'source', 'post_dttm', 'data']
    ].head(30)
)

In [ ]:
#%% md
## Семантическая гипотеза — ModernBERT


#%%
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer

# Загружаем только токенайзер — модель и GPU для этой проверки не нужны
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Считаем число токенов без truncation, батчами
token_lengths = []

for start in tqdm(
    range(0, len(df), 256),
    desc='Подсчёт токенов'
):
    texts_batch = df['text_normalized'].iloc[start:start + 256].tolist()

    encoded = tokenizer(
        texts_batch,
        add_special_tokens=True,
        truncation=False,
        padding=False
    )
    
    token_lengths.extend(len(ids) for ids in encoded['input_ids'])

df['token_length'] = token_lengths

# Ключевые квантили: покажут разумный max_length
quantiles = df['token_length'].quantile(
    [0.50, 0.75, 0.90, 0.95, 0.99, 1.00]
).round().astype(int)

display(quantiles.rename('число токенов').to_frame())

print(
    f"Постов, обрезаемых при max_length=512:  {(df['token_length'] > 512).mean():.2%}\n"
    f"Постов, обрезаемых при max_length=1024: {(df['token_length'] > 1024).mean():.2%}\n"
    f"Постов, обрезаемых при max_length=2048: {(df['token_length'] > 2048).mean():.2%}"
)
#%%


In [ ]:
# Основное распределение — ограничено 99-м перцентилем,
# чтобы несколько очень длинных постов не «сжали» график.
p99 = df['token_length'].quantile(0.99)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(
    df.loc[df['token_length'] <= p99, 'token_length'],
    bins=60,
    ax=axes[0]
)
axes[0].axvline(512, color='orange', linestyle='--', label='512')
axes[0].axvline(1024, color='red', linestyle='--', label='1024')
axes[0].set(
    title='Распределение длины новостей в токенах (до 99-го перцентиля)',
    xlabel='Токенов',
    ylabel='Количество новостей'
)
axes[0].legend()

# Кумулятивная доля: удобна для выбора max_length
sorted_lengths = np.sort(df['token_length'])
cdf = np.arange(1, len(sorted_lengths) + 1) / len(sorted_lengths)

axes[1].plot(sorted_lengths, cdf, linewidth=2)
for length in (256, 512, 1024, 2048):
    share = (df['token_length'] <= length).mean()
    axes[1].axvline(length, color='gray', linestyle='--', alpha=0.6)
    axes[1].annotate(
        f'{length}: {share:.1%}',
        xy=(length, share),
        xytext=(length, min(share + 0.06, 0.98)),
        ha='center'
    )

axes[1].set(
    title='Накопленная доля покрытых новостей',
    xlabel='max_length',
    ylabel='Доля новостей без обрезания',
    ylim=(0, 1.05)
)
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()
#%%


In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

def modernbert_embeddings(texts, batch_size=128):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Устройство: {device}; модель: {MODEL_NAME}')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()
    result = []
    with torch.inference_mode():
        for start in tqdm(range(0, len(texts), batch_size), desc='ModernBERT: подготовка текстов'):
            batch = tokenizer(texts[start:start + batch_size], padding=True, truncation=True,
                              max_length=1024, return_tensors='pt').to(device)
            hidden = model(**batch).last_hidden_state
            mask = batch['attention_mask'].unsqueeze(-1)
            pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1)
            result.append(torch.nn.functional.normalize(pooled, p=2, dim=1).cpu().numpy())
    return np.vstack(result)

emb_path = OUT_DIR / 'modernbert_embeddings.npy'
if emb_path.exists():
    embeddings = np.load(emb_path)
else:
    embeddings = modernbert_embeddings(df.text_normalized.tolist())
    np.save(emb_path, embeddings)
embeddings.shape

In [ ]:

# Чем выше порог, тем меньше риск склеить разные новости
# ============================================================================
# АДАПТИВНЫЙ ПОИСК ОПТИМАЛЬНОГО ПОРОГА ДЕДУПЛИКАЦИИ
# ============================================================================
# Этот код заменяет ручной перебор порогов (0.87, 0.90, 0.93)
# Он автоматически находит лучший порог на основе метрики удалённых новостей.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score
from scipy.signal import savgol_filter

# Вспомогательные функции (они уже есть в ноутбуке, но продублируем для ясности)
class UnionFind:
    def __init__(self, n): self.parent = list(range(n))
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b: self.parent[b] = a

def clusters_from_edges(n, edges):
    uf = UnionFind(n)
    for a, b in edges: uf.union(a, b)
    return pd.factorize([uf.find(i) for i in range(n)])[0]

# ИСПРАВЛЕНО: раньше здесь был пустой заглушечный top_k_edges (только `pass`),
# который возвращал None и ломал весь адаптивный поиск. Возвращена рабочая реализация.
def top_k_edges(vectors, dates, threshold, n_neighbors=6, batch_size=128):
    ts = pd.to_datetime(dates, utc=True).astype('int64').to_numpy()
    max_delta = pd.Timedelta(days=WINDOW_DAYS).value
    edges = []
    for start in tqdm(range(0, len(ts), batch_size), desc=f'Поиск пар ≥ {threshold:.2f}', leave=False):
        stop = min(start + batch_size, len(ts))
        sim = vectors[start:stop] @ vectors.T
        if hasattr(sim, 'toarray'): sim = sim.toarray()
        idx = np.argpartition(sim, -n_neighbors, axis=1)[:, -n_neighbors:]
        for local_i, candidates in enumerate(idx):
            i = start + local_i
            for j in candidates:
                if i < j and sim[local_i, j] >= threshold and abs(ts[i] - ts[j]) <= max_delta:
                    edges.append((i, int(j)))
    return edges

def compute_duplicate_stats(threshold, embeddings, dates, top_k=6):
    edges = top_k_edges(embeddings, dates, threshold, n_neighbors=top_k)
    clusters = clusters_from_edges(len(df), edges)
    sizes = pd.Series(clusters).value_counts()
    dup_sizes = sizes[sizes > 1]
    num_dup_clusters = len(dup_sizes)
    total_duplicates = dup_sizes.sum()
    removed = (dup_sizes - 1).sum()
    avg_dup_size = dup_sizes.mean() if len(dup_sizes) > 0 else 0
    # ИСПРАВЛЕНО: раньше маска clusters != -1 фактически включала ВСЕ ~19 тыс. точек
    # (включая одиночные кластеры), из-за чего silhouette_score считался по всему
    # датасету на каждой итерации поиска — это O(n^2) и практически не завершается.
    # Теперь silhouette считается только по новостям из кластеров-дублей (size > 1).
    if len(dup_sizes) > 1:
        dup_mask = np.isin(clusters, dup_sizes.index)
        if dup_mask.sum() > 1:
            X_clean = embeddings[dup_mask]
            labels_clean = clusters[dup_mask]
            try:
                sil = silhouette_score(X_clean, labels_clean, metric='cosine')
            except ValueError:
                sil = -1
        else:
            sil = -1
    else:
        sil = -1
    return {'threshold': threshold, 'num_dup_clusters': num_dup_clusters,
            'total_duplicates': total_duplicates, 'removed': removed,
            'avg_dup_size': avg_dup_size, 'silhouette': sil, 'clusters': clusters}

def adaptive_threshold_search(embeddings, dates,
                              initial_step=0.1,
                              min_step=0.001,
                              change_threshold=0.05,
                              max_iter=50,
                              top_k=6):
    thresholds = [0.0, 0.5, 1.0]
    results = {}
    for t in thresholds:
        results[t] = compute_duplicate_stats(t, embeddings, dates, top_k)
    intervals = [(0.0, 0.5), (0.5, 1.0)]
    iteration = 0
    while intervals and iteration < max_iter:
        new_intervals = []
        for t1, t2 in intervals:
            if t2 - t1 <= min_step:
                continue
            mid = (t1 + t2) / 2
            if mid not in results:
                results[mid] = compute_duplicate_stats(mid, embeddings, dates, top_k)
            r1 = results[t1]['removed']
            r2 = results[t2]['removed']
            r_mid = results[mid]['removed']
            delta = abs(r2 - r1) / max(1, abs(r1))
            if delta > change_threshold:
                new_intervals.append((t1, mid))
                new_intervals.append((mid, t2))
            delta_left = abs(r_mid - r1) / max(1, abs(r1))
            if delta_left > change_threshold:
                new_intervals.append((t1, mid))
            delta_right = abs(r2 - r_mid) / max(1, abs(r_mid))
            if delta_right > change_threshold:
                new_intervals.append((mid, t2))
        intervals = new_intervals
        iteration += 1
    df_results = pd.DataFrame([
        {'threshold': t, 'num_dup_clusters': res['num_dup_clusters'],
         'total_duplicates': res['total_duplicates'], 'removed': res['removed'],
         'avg_dup_size': res['avg_dup_size'], 'silhouette': res['silhouette']}
        for t, res in results.items()
    ]).sort_values('threshold').reset_index(drop=True)
    return df_results, results

def find_optimal_threshold(df_results, metric='removed', smooth_window=5):
    if len(df_results) > smooth_window:
        y = savgol_filter(df_results[metric].values, window_length=smooth_window, polyorder=2)
    else:
        y = df_results[metric].values
    x = df_results['threshold'].values
    dy = np.gradient(y, x)
    d2y = np.gradient(dy, x)
    idx_max = np.argmax(d2y)
    optimal_threshold = x[idx_max]
    return optimal_threshold, df_results.iloc[idx_max]

# --- ЗАПУСК АДАПТИВНОГО ПОИСКА ---
print("Запуск адаптивного поиска порога...")
df_adaptive, all_results = adaptive_threshold_search(
    embeddings, df.post_dttm,
    initial_step=0.1,
    min_step=0.001,
    change_threshold=0.05,
    max_iter=30,
    top_k=6
)

optimal_th, optimal_row = find_optimal_threshold(df_adaptive, metric='removed', smooth_window=5)
# ИСПРАВЛЕНО: прямое обращение all_results[optimal_th] могло не найти ключ из-за
# погрешности float. Берём ближайший реально посчитанный порог.
optimal_key = min(all_results.keys(), key=lambda k: abs(k - optimal_th))
print(f"Оптимальный порог (по точке перегиба): {optimal_th:.3f}")
print(f"При этом пороге: удалено {optimal_row['removed']:.0f} новостей, "
      f"кластеров дублей: {optimal_row['num_dup_clusters']:.0f}, "
      f"силуэт: {optimal_row['silhouette']:.3f}")

# --- ВИЗУАЛИЗАЦИЯ ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
ax = axes[0, 0]
ax.plot(df_adaptive['threshold'], df_adaptive['removed'], 'o-', label='removed')
ax.axvline(optimal_th, color='red', linestyle='--', label=f'оптимальный {optimal_th:.3f}')
ax.set_xlabel('Порог')
ax.set_ylabel('Удалённых новостей')
ax.legend()
ax.grid(True)

ax = axes[0, 1]
ax.plot(df_adaptive['threshold'], df_adaptive['avg_dup_size'], 's-', color='green', label='avg cluster size')
ax.axvline(optimal_th, color='red', linestyle='--')
ax.set_xlabel('Порог')
ax.set_ylabel('Средний размер кластера дублей')
ax.legend()
ax.grid(True)

ax = axes[1, 0]
ax.plot(df_adaptive['threshold'], df_adaptive['num_dup_clusters'], '^-', color='orange', label='num clusters')
ax.axvline(optimal_th, color='red', linestyle='--')
ax.set_xlabel('Порог')
ax.set_ylabel('Количество кластеров дублей')
ax.legend()
ax.grid(True)

ax = axes[1, 1]
mask = df_adaptive['silhouette'] > -0.5
ax.plot(df_adaptive.loc[mask, 'threshold'], df_adaptive.loc[mask, 'silhouette'], 'd-', color='purple', label='silhouette')
ax.axvline(optimal_th, color='red', linestyle='--')
ax.set_xlabel('Порог')
ax.set_ylabel('Silhouette score')
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.savefig('adaptive_threshold_analysis.png', dpi=150)
plt.show()

df_adaptive.to_csv('adaptive_threshold_results.csv', index=False, encoding='utf-8-sig')
print("Результаты поиска сохранены в adaptive_threshold_results.csv")

# --- ИСПОЛЬЗУЕМ НАЙДЕННЫЙ ПОРОГ ДЛЯ ДЕДУПЛИКАЦИИ ---
# Теперь мы можем использовать optimal_th вместо фиксированных порогов.
experiments[f'modernbert_adaptive_{optimal_th:.3f}'] = all_results[optimal_key]['clusters']
print(f"Добавлен кластер для порога {optimal_th:.3f} в словарь experiments")

In [ ]:
# Ручная проверка: посмотрите кластеры выбранного метода до удаления.
# ИСПРАВЛЕНО: раньше SELECTED_METHOD = 'modernbert_0.93' ссылался на ключ,
# которого не было в словаре experiments (такой порог никогда не считался) -> KeyError.
# Теперь используем ключ, реально добавленный адаптивным поиском в предыдущей ячейке.
SELECTED_METHOD = f'modernbert_adaptive_{optimal_th:.3f}'
df['duplicate_cluster'] = experiments[SELECTED_METHOD]
cluster_sizes = df.duplicate_cluster.value_counts()
sample_ids = cluster_sizes[cluster_sizes > 1].head(10).index
display(df[df.duplicate_cluster.isin(sample_ids)]
        .sort_values(['duplicate_cluster', 'post_dttm'])
        [['duplicate_cluster', 'source', 'post_dttm', 'data']])

In [ ]:
# Размеры кластеров, где найдено больше одной похожей новости
cluster_sizes = df['duplicate_cluster'].value_counts()
duplicate_clusters = cluster_sizes[cluster_sizes > 1]

print(f'Кластеров со схожими новостями: {len(duplicate_clusters)}')
print(f'Новостей внутри таких кластеров: {duplicate_clusters.sum()}')

# Выберите кластер для детального просмотра
# ИСПРАВЛЕНО: index[4] мог выйти за границы, если найдено меньше 5 кластеров-дублей.
cluster_id = duplicate_clusters.index[min(4, len(duplicate_clusters) - 1)]

print(f'Кластер {cluster_id}, новостей: {duplicate_clusters[cluster_id]}')

display(
    df.loc[df['duplicate_cluster'] == cluster_id,
           ['id', 'source', 'post_dttm', 'data']]
      .sort_values('post_dttm')
)

In [ ]:
# Посты, которые будут исключены: все, кроме самой ранней новости в каждом кластере.
ordered = df.sort_values(['duplicate_cluster', 'post_dttm']).copy()
posts_to_remove = ordered[ordered.duplicated('duplicate_cluster', keep='first')].copy()

print(f'К исключению: {len(posts_to_remove):,} постов')
display(posts_to_remove[['duplicate_cluster', 'id', 'source', 'post_dttm', 'data']]
        .sort_values(['duplicate_cluster', 'post_dttm'])
        .head(30))

# Все строки таблицы можно открыть отдельно в Excel или в pandas.
posts_to_remove.to_csv(OUT_DIR / f'posts_to_remove_{SELECTED_METHOD}.csv',
                       index=False, encoding='utf-8-sig')

In [ ]:
# ИСПРАВЛЕНО: faiss не гарантированно установлен в окружении - добавлен безопасный fallback.
try:
    import faiss
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu'])
    import faiss
import numpy as np

# embeddings — эмбеддинги ModernBERT формы (число_новостей, размер_эмбеддинга)
# Например, embeddings = modernbert_embeddings(df.text_normalized.tolist())
embeddings = embeddings.astype(np.float32)

# Нормализация: скалярное произведение станет cosine similarity
faiss.normalize_L2(embeddings)

TOP_K = 2
SIMILARITY_THRESHOLD = 0.90

# Для 19 тыс. новостей точный поиск — хороший и быстрый вариант
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

# similarities[i] — cosine similarity новости i с соседями
# neighbor_ids[i] — их индексы в df
similarities, neighbor_ids = index.search(embeddings, TOP_K)

# Формируем пары семантических дублей
edges = []

for i in range(len(df)):
    for similarity, j in zip(similarities[i, 1:], neighbor_ids[i, 1:]):
        if i < j and similarity >= SIMILARITY_THRESHOLD:
            edges.append((i, int(j)))

# Объединяем связанные пары в кластеры
faiss_clusters = clusters_from_edges(len(df), edges)

# Добавляем номера кластеров в датафрейм
df_faiss = df.assign(duplicate_cluster=faiss_clusters)

# Оставляем первую публикацию из каждого кластера
news_deduplicated = (
    df_faiss.sort_values('post_dttm')
    .drop_duplicates('duplicate_cluster', keep='first')
)

# Посты, которые будут исключены
posts_to_remove = (
    df_faiss.sort_values(['duplicate_cluster', 'post_dttm'])
    .loc[lambda x: x.duplicated('duplicate_cluster', keep='first')]
)

print(f'Найдено пар дублей: {len(edges):,}')
print(f'Было постов: {len(df_faiss):,}')
print(f'К удалению: {len(posts_to_remove):,}')
print(f'Осталось: {len(news_deduplicated):,}')

display(posts_to_remove[
    ['duplicate_cluster', 'source', 'post_dttm', 'data']
].head(30))

In [ ]:
# Сохраняем раннюю публикацию из каждого кластера и все найденные кластеры для аудита.
deduplicated = (df.sort_values('post_dttm')
                  .drop_duplicates('duplicate_cluster', keep='first')
                  .drop(columns=['duplicate_cluster']))
duplicates_audit = (df[df.duplicate_cluster.duplicated(keep=False)]
                    .sort_values(['duplicate_cluster', 'post_dttm']))

deduplicated.to_csv(OUT_DIR / f'news_deduplicated_{SELECTED_METHOD}.csv', index=False, encoding='utf-8-sig')
duplicates_audit.to_csv(OUT_DIR / f'duplicate_clusters_{SELECTED_METHOD}.csv', index=False, encoding='utf-8-sig')
report.to_csv(OUT_DIR / 'hypotheses_comparison.csv', index=False, encoding='utf-8-sig')
print(f'Было: {len(df):,}; осталось: {len(deduplicated):,}; удалено: {len(df) - len(deduplicated):,}')

In [ ]:
import os
# Создаём папку для вывода
os.makedirs('/kaggle/working/output', exist_ok=True)
# Сохраняем CSV
df_adaptive.to_csv('/kaggle/working/adaptive_threshold_results.csv', index=False)

In [ ]:
# Проверка существования переменной
try:
    print(df_adaptive.head())
except NameError:
    print("❌ Переменная df_adaptive не определена.")

In [ ]:
#%% md
## 6. Распределение новостей по дням
#Смотрим, как менялось число публикаций по дням — до фильтрации (весь df) и после
#дедупликации выбранным методом (SELECTED_METHOD, датафрейм `deduplicated` из раздела сохранения).
#%%
daily_counts_raw = df.assign(day=df.post_dttm.dt.date).groupby('day').size()
daily_counts_dedup = deduplicated.assign(day=deduplicated.post_dttm.dt.date).groupby('day').size()

daily = pd.DataFrame({
    'до фильтрации': daily_counts_raw,
    'после фильтрации': daily_counts_dedup
}).fillna(0).sort_index()

fig, ax = plt.subplots(figsize=(16, 5))
daily.plot(ax=ax, linewidth=1.5)
ax.set_title(f'Число новостей по дням до и после дедупликации ({SELECTED_METHOD})')
ax.set_xlabel('Дата')
ax.set_ylabel('Число новостей')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('daily_news_distribution.png', dpi=150)
plt.show()

daily.to_csv(OUT_DIR / 'daily_news_distribution.csv', encoding='utf-8-sig')
daily.tail()

In [ ]:
#%% md
## 7. Распределение новостей по тикерам
#Извлекаем упоминания тикеров MOEX из текста новостей и считаем частоту по каждому тикеру.
#Список актуальных тикеров берём через публичный MOEX ISS API (без ключа); если он недоступен —
#используем запасной список наиболее ликвидных бумаг.
#%%
import requests

def load_moex_tickers():
    """Список тикеров акций MOEX (SECID) для сопоставления с текстом новостей."""
    url = 'https://iss.moex.com/iss/engines/stock/markets/shares/securities.json'
    try:
        resp = requests.get(url, params={'iss.meta': 'off'}, timeout=20)
        resp.raise_for_status()
        payload = resp.json()['securities']
        secid_idx = payload['columns'].index('SECID')
        tickers = sorted({row[secid_idx] for row in payload['data'] if row[secid_idx]})
        if tickers:
            return tickers
        raise ValueError('пустой список тикеров от MOEX ISS')
    except Exception as exc:
        print(f'Не удалось получить список тикеров с MOEX ISS ({exc}), использую запасной список.')
        return ['SBER', 'GAZP', 'LKOH', 'GMKN', 'NVTK', 'ROSN', 'TATN', 'MGNT',
                'MTSS', 'YDEX', 'VTBR', 'CHMF', 'ALRS', 'PLZL', 'MOEX', 'AFLT',
                'PHOR', 'RUAL', 'POLY', 'SNGS']

MOEX_TICKERS = load_moex_tickers()
print(f'Загружено тикеров: {len(MOEX_TICKERS)}')

# Ищем тикеры в исходном тексте 'data' (а не text_normalized — там всё в нижнем регистре,
# и совпадения по коротким тикерам типа 'MGNT' стали бы неотличимы от обычных слов).
ticker_pattern = re.compile(
    r'(?:\$)?\b(' + '|'.join(re.escape(t) for t in MOEX_TICKERS) + r')\b'
)

def extract_tickers(text):
    return set(ticker_pattern.findall(str(text or '')))

df['tickers_found'] = df['data'].map(extract_tickers)
ticker_counts = pd.Series(
    [t for tickers in df['tickers_found'] for t in tickers]
).value_counts()

covered = (df['tickers_found'].map(len) > 0).sum()
print(f'Новостей с хотя бы одним тикером: {covered:,} из {len(df):,} ({covered / len(df):.1%})')

top_n = min(25, len(ticker_counts))
if top_n > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, top_n * 0.3)))
    ticker_counts.head(top_n).sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Топ-{top_n} тикеров по числу упоминаний в новостях')
    ax.set_xlabel('Число новостей')
    plt.tight_layout()
    plt.savefig('ticker_distribution.png', dpi=150)
    plt.show()
    ticker_counts.to_csv(OUT_DIR / 'ticker_distribution.csv', header=['count'], encoding='utf-8-sig')
else:
    print('Тикеры в тексте не найдены — проверьте формат данных или список тикеров.')

In [ ]:
#%% md
## 8. Регрессионный анализ: новостной поток vs рынок (до и после фильтрации)
#Берём дневные котировки индекса MOEX (IMOEX) через публичный ISS API и смотрим,
#насколько число новостей в день объясняет дневную волатильность (|доходность|) —
#отдельно на сыром потоке и после дедупликации. Сравнение R² до/после — количественная
#иллюстрация того, что даёт фильтрация дублей для последующего анализа.
#%%
try:
    import statsmodels.api as sm
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'statsmodels'])
    import statsmodels.api as sm

def load_moex_index_history(secid='IMOEX', board='SNDX', start=None, end=None):
    """Дневная история индекса MOEX через публичный ISS API (без ключа, постранично)."""
    base = f'https://iss.moex.com/iss/history/engines/stock/markets/index/boards/{board}/securities/{secid}.json'
    frames = []
    start_pos = 0
    while True:
        params = {'iss.meta': 'off', 'start': start_pos}
        if start: params['from'] = start
        if end: params['till'] = end
        resp = requests.get(base, params=params, timeout=20)
        resp.raise_for_status()
        payload = resp.json()['history']
        rows = payload['data']
        if not rows:
            break
        frames.append(pd.DataFrame(rows, columns=payload['columns']))
        start_pos += len(rows)
        if len(rows) < 100:
            break
    if not frames:
        return pd.DataFrame()
    hist = pd.concat(frames, ignore_index=True)
    hist['TRADEDATE'] = pd.to_datetime(hist['TRADEDATE'])
    return hist[['TRADEDATE', 'CLOSE']].dropna().drop_duplicates('TRADEDATE')

market_start = df.post_dttm.min().date().isoformat()
market_end = df.post_dttm.max().date().isoformat()

try:
    market = load_moex_index_history('IMOEX', 'SNDX', market_start, market_end)
except Exception as exc:
    print(f'Не удалось получить рыночные данные с MOEX ISS: {exc}')
    market = pd.DataFrame()

if market.empty:
    print('Регрессия пропущена: нет рыночных данных (нет интернета или MOEX ISS недоступен).')
else:
    market = market.sort_values('TRADEDATE').reset_index(drop=True)
    market['return'] = market['CLOSE'].pct_change()
    market['abs_return'] = market['return'].abs()
    market['day'] = market['TRADEDATE'].dt.date

    reg_df = market[['day', 'abs_return']].merge(
        daily.rename_axis('day').reset_index(), on='day', how='inner'
    ).dropna()

    def run_ols(x_col, label):
        X = sm.add_constant(reg_df[[x_col]])
        y = reg_df['abs_return']
        model = sm.OLS(y, X).fit()
        print(f'--- {label} (X = {x_col}) ---')
        print(f"R^2 = {model.rsquared:.4f}; coef = {model.params[x_col]:.6f}; "
              f"p-value = {model.pvalues[x_col]:.4f}; n = {int(model.nobs)}")
        return model

    if len(reg_df) < 5:
        print(f'Слишком мало пересекающихся торговых дней ({len(reg_df)}) для регрессии.')
    else:
        model_raw = run_ols('до фильтрации', 'До фильтрации (сырой новостной поток)')
        model_dedup = run_ols('после фильтрации', 'После фильтрации (дедупликация)')

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        for ax, (x_col, model, title) in zip(
            axes,
            [('до фильтрации', model_raw, 'До фильтрации'),
             ('после фильтрации', model_dedup, 'После фильтрации')]
        ):
            ax.scatter(reg_df[x_col], reg_df['abs_return'], alpha=0.5, s=15)
            xs = np.linspace(reg_df[x_col].min(), reg_df[x_col].max(), 50)
            ys = model.params['const'] + model.params[x_col] * xs
            ax.plot(xs, ys, color='red')
            ax.set_title(f'{title}: R\u00b2={model.rsquared:.3f}')
            ax.set_xlabel('Число новостей в день')
            ax.set_ylabel('|Дневная доходность IMOEX|')
            ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig('regression_news_vs_market.png', dpi=150)
        plt.show()

        reg_df.to_csv(OUT_DIR / 'regression_news_vs_market.csv', index=False, encoding='utf-8-sig')

In [ ]:
#%% md
## 9. Идеи для развития проекта (по заметкам с созвона с куратором)
#
# 1. Непрерывная метрика новизны вместо бинарного дубль/не дубль — по аналогии с
#    Event Similarity Gap / Event Novelty Score у RavenPack: число дней с последнего
#    похожего события по тому же тикеру, используется как фича, а не только для чистки.
#    RavenPack, "Improved Stock Market Returns From Systematically Trading Infrequent News".
#
# 2. Проверить гипотезу "stale news" (Tetlock, 2011, AFA/SSRN "All the News That's Fit
#    to Reprint"): реагирует ли рынок слабее на новость, похожую на недавние по тому же
#    тикеру, и есть ли разворот доходности после таких "устаревших" публикаций.
#
# 3. LLM-усиленная кластеризация событий вместо чистого cosine-threshold: использовать
#    LLM для проверки/уточнения кластеров дублей и метрику типа CSAI (Cluster Stability
#    Assessment Index) для оценки качества. arXiv:2406.10552,
#    "Large Language Model Enhanced Clustering for News Event Detection".
#
# 4. Иерархические Matryoshka-эмбеддинги для two-stage поиска: сначала грубая
#    кластеризация по укороченному вектору (тема/сюжет), затем точный поиск дублей на
#    полноразмерном векторе только внутри кандидатов — экономит вычисления по сравнению
#    с полным FAISS-поиском по всем векторам. arXiv:2506.00277, "Hierarchical Level-Wise
#    News Article Clustering via Multilingual Matryoshka Embeddings".
#
# 5. Потоковая (инкрементальная) дедупликация вместо батчевой обработки всего датасета
#    разом: двухэтапная схема LSH-блокировка + точная классификация пар, актуально для
#    реального Telegram-потока в проде. "Двухшаговый метод объединения новостей в
#    сюжеты" (КиберЛенинка).
#
# 6. Лёгкие статические эмбеддинги (Model2Vec/SemHash) для первого грубого прохода
#    дедупликации на масштабе — на порядок быстрее transformer-энкодера, полезно для
#    near-real-time сценария и как baseline перед ModernBERT.
#
# 7. Сравнение нескольких эмбеддинг-моделей на RU-корпусе (RuModernBERT-small vs
#    BGE-M3 / E5-подобные multi-granularity модели) — качество semantic dedup сильно
#    зависит от модели, стоит явно замерить.
#
# 8. Разделить "дубль по тикеру" и "дубль по событию, но без общего тикера" — событийно
#    ориентированный подход (аналог REL/SIM в RavenPack): relevance-score, где упомянут
#    тикер, плюс отдельная кластеризация по событию.
#
# 9. Regression по волатильности/объёму торгов, а не только |доходности| — реализованная
#    внутридневная волатильность и объём чувствительнее к информационному потоку.
#
# 10. Attention spillover: учитывать не только новости по конкретному тикеру, но и по
#     связанным бумагам того же сектора — они тоже двигают цену. "Joint News, Attention
#     Spillover, and Market Returns" (Li Guo, Fudan University).
#%%
print('См. текстовые заметки выше — раздел с идеями для дальнейшего развития проекта.')